# Exploratory Data Analysis

Let's look at the dataset structure then load sample data and accomodate with the data structure.

In [29]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [30]:
raw_kmt_dataset = Path("../data/raw_kmt_dataset")
json_file_list = [path.name for path in raw_kmt_dataset.glob("*.json")]
json_file_list.sort()
json_file_list[:5]

['raw_kmt_user_0001.json',
 'raw_kmt_user_0002.json',
 'raw_kmt_user_0003.json',
 'raw_kmt_user_0004.json',
 'raw_kmt_user_0005.json']

In RAW KMT Dataset directory there are JSON files representing datasets for each user.
Let's look at the JSON structure.

In [31]:
with open(raw_kmt_dataset / json_file_list[0], 'r') as f:
    user1 = json.load(f)

print(user1.keys())

dict_keys(['details', 'true_data', 'false_data'])


In [32]:
print(user1["details"])
print(user1["false_data"].keys())
print(user1["true_data"].keys())

{'ID': 'CDID0001', 'Provider': 'Discover', 'Name': 'Mrs Ellie Miller', 'Card Number': 4422891459068728, 'CVC': 646, 'Expiry': '09/23'}
dict_keys(['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'test_6', 'test_7', 'test_8', 'test_9', 'test_10'])
dict_keys(['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'test_6', 'test_7', 'test_8', 'test_9', 'test_10'])


In [33]:
print(user1["true_data"]["test_1"].keys())

dict_keys(['key_events', 'mouse_events'])


In [34]:
user1_key = pd.DataFrame(user1["true_data"]["test_1"]["key_events"])

display(user1_key)
display(user1_key.describe(include="all"))
display(user1_key.dtypes)

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
0,shift,pressed,Name,True,2022-02-06 18:57:09.981740,1644173829.9817405
1,m,pressed,Name,True,2022-02-06 18:57:10.121021,1644173830.1210213
2,shift,released,Name,True,2022-02-06 18:57:10.169470,1644173830.1694703
3,m,released,Name,True,2022-02-06 18:57:10.218179,1644173830.21818
4,r,pressed,Name,True,2022-02-06 18:57:10.293582,1644173830.293582
...,...,...,...,...,...,...
113,tab,released,Exp y,True,2022-02-06 18:57:26.707177,1644173846.7071774
114,2,pressed,Exp y,True,2022-02-06 18:57:26.834916,1644173846.8349168
115,2,released,Exp y,True,2022-02-06 18:57:26.895162,1644173846.8951626
116,3,pressed,Exp y,True,2022-02-06 18:57:26.958561,1644173846.9585614


,Key,Event,Input Box,Text Changed,Timestamp,Epoch
count,118,118,118,118,118,118
unique,21,2,5,2,116,116
top,backspace,released,Name,True,2022-02-06 18:57:10.756190,1644173830.7561908
freq,14,61,64,112,2,2


Key              str
Event            str
Input Box        str
Text Changed    bool
Timestamp        str
Epoch            str
dtype: object

## Keyboard data
The keyboard data is was collected at the event level.

Six features were registered:
- `Key` - self-explainatory
- `Event` - either *press* or *release* event type
- `Input Box` - which text box of the GUI app the user is currently typing in
- `Text Changed` - if the event has changed the data in the text field or not (eg. typing in more than three digits in the CVC field will not change the text field)
- `Timestamp` - Date and Time with us precision
- `Epoch` - seconds elapsed from the set date (1-1-1970)

`Timestamp` and `Epoch` columns duplicate the same information. One of them can be dropped.

Consider dropping `shift` key as well as other possible modifier keys as they are pressed and held for a longer time.

**NOTE**<br>
Differentiating between using `tab` and/or `left-click` for moving between the fields on the form may have discriminatory power.
`shift` as well as other modifier keys are pressed for a substantially longer time than alphanumeric keys which may be of use. 

Let's now look at the whole concatenated keyboard event dataset to check for other possible discriminating features and potential issues with the data.

In [35]:
dfs = []
for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for no in range(1, 11):
            dfs.append(pd.DataFrame(user_data["true_data"][f"test_{no}"]["key_events"]))
            dfs.append(pd.DataFrame(user_data["false_data"][f"test_{no}"]["key_events"]))

big_keyboard_df = pd.concat(dfs).astype({
    "Epoch": "Float64"
})

display(big_keyboard_df.describe(include="all"))
display(big_keyboard_df.isna().sum())

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
count,172563,172563,172563,172563,172563,172563.0
unique,82,2,6,2,171463,<NA>
top,backspace,pressed,Name,True,2022-03-01 13:48:50.519671,<NA>
freq,7925,87608,79905,162836,4,<NA>
mean,NaN,NaN,NaN,NaN,NaN,1647302420.65725
std,NaN,NaN,NaN,NaN,NaN,1935228.31744
min,NaN,NaN,NaN,NaN,NaN,1643113826.341281
25%,NaN,NaN,NaN,NaN,NaN,1646151579.757441
50%,NaN,NaN,NaN,NaN,NaN,1647353218.034949
75%,NaN,NaN,NaN,NaN,NaN,1648984188.621257


Key             0
Event           0
Input Box       0
Text Changed    0
Timestamp       0
Epoch           0
dtype: int64

- Overall over 170,000 key events were registered in the experiments. 
- They represent 82 unique keys. 
- Only `pressed` and `released` events were collected. No issues there so far.
- There is no missing data in any of the features.

In [36]:
big_keyboard_df["Key"].unique()

<ArrowStringArray>
[          'shift',               'm',               'r',               's',
        'spacebar',               'w',               'e',       'backspace',
               'l',               'i',             'tab',               '4',
               '2',               '8',               '9',               '1',
               '5',               '0',               '6',               '7',
               '3',          'rshift',         'numpad4',         'numpad2',
         'numpad8',         'numpad9',         'numpad1',         'numpad5',
         'numpad0',         'numpad6',         'numpad7',         'numpad3',
        'capslock', 'numpadsubstract',               '.',               'j',
               ',',               'h',               'd',               't',
               'a',               'y',               'u',               'o',
         'numlock',               'f',           'enter',               'n',
               'g',               '-',               '`',

**NOTE**

Possible discrimination points:
- `NUMPAD` key usage
- using `TAB` and/or mouse `LEFT CLICK` for moving between text fieldsd
- `SHIFT` and/or `CAPSLOCK` and using either left or right `SHIFT`
- making typing mistakes and correcting them with `BACKSPACE` and/or `DELETE`
- using `ARROW` keys for navigation

In [37]:
big_keyboard_df["Event"].value_counts()

Event
pressed     87608
released    84955
Name: count, dtype: int64

**NOTE**<br>
The above result suggests that not all key events are paired which implies the dataset has missing events and/or data collection quirks exist.

Let's look a little deeper into that.

In [38]:
big_keyboard_df.groupby(["Key", "Event"]).count()

Input Box  Text Changed  Timestamp  Epoch
Key Event                                              
#   pressed          22            22         22     22
    released         22            22         22     22
'   pressed           1             1          1      1
    released          1             1          1      1
,   pressed          11            11         11     11
...                 ...           ...        ...    ...
x   released        106           106        106    106
y   pressed         613           613        613    613
    released        613           613        613    613
z   pressed          66            66         66     66
    released         66            66         66     66

[164 rows x 4 columns]

There are small anomalies in shortly pressed keys like `2` and `8` and much bigger discrepancies for `backspace`, `shift`, `rshift`, `spacebar` and more. In all of these instances there are more `pressed` events than `released`, which suggests that a long press may be logged as consecutive `pressed` events.

However, something particularly insteresting happens for `Tab` key with an overwhelming predominance of `released` events.

Let's look into that next.

In [39]:
shifted_df = big_keyboard_df.groupby("Key", sort=False)[["Key","Event", "Epoch"]].shift(-1)
shifted_df.columns = ["Key_shifted", "Event_shifted", "Epoch_shifted"]
comparison_df = pd.concat([big_keyboard_df, shifted_df], axis=1)
comparison_df.loc[(comparison_df["Key"] == "2") & (comparison_df["Event"] == comparison_df["Event_shifted"])]
comparison_df[comparison_df["Epoch"].between(1648896881.0545242 - 5, 1648896881.0545242 + 30)]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch,Key_shifted,Event_shifted,Epoch_shifted
70,4,released,Card No,True,2022-04-02 10:54:36.271858,1648896876.271858,4,pressed,1649073768.692647
71,5,pressed,Card No,True,2022-04-02 10:54:36.418063,1648896876.418063,5,released,1648896876.577644
72,5,released,Card No,True,2022-04-02 10:54:36.577644,1648896876.577644,5,pressed,1648896885.870337
73,7,pressed,Card No,True,2022-04-02 10:54:37.261808,1648896877.261808,7,released,1648896877.469558
74,7,released,Card No,True,2022-04-02 10:54:37.469557,1648896877.469558,7,pressed,1648896897.060272
75,1,pressed,Card No,True,2022-04-02 10:54:37.592983,1648896877.592984,1,released,1648896877.759992
76,1,released,Card No,True,2022-04-02 10:54:37.759991,1648896877.759992,1,pressed,1648896886.395125
77,9,pressed,Card No,True,2022-04-02 10:54:40.169949,1648896880.16995,9,released,1648896880.319356
78,9,released,Card No,True,2022-04-02 10:54:40.319356,1648896880.319356,9,pressed,1649073779.429939
79,2,pressed,Card No,True,2022-04-02 10:54:40.796559,1648896880.797569,2,pressed,1648896881.054524


It one peculiar episode the `2` key was pressed three times before relesase after which the user has corrected himself deleting two of the `2`. Perhaps, he pressed for too long or some app glitch happened and he did not see the result?

In [40]:
big_keyboard_df.loc[big_keyboard_df["Key"] == "shift"]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
0,shift,pressed,Name,True,2022-02-06 18:57:09.981740,1644173829.98174
2,shift,released,Name,True,2022-02-06 18:57:10.169470,1644173830.16947
9,shift,pressed,Name,True,2022-02-06 18:57:10.679335,1644173830.679336
15,shift,released,Name,True,2022-02-06 18:57:10.844363,1644173830.844363
20,shift,pressed,Name,True,2022-02-06 18:57:11.721636,1644173831.721636
...,...,...,...,...,...,...
26,shift,pressed,Name,True,2022-04-14 09:59:00.636272,1649930340.636272
27,shift,pressed,Name,True,2022-04-14 09:59:00.661282,1649930340.661283
28,shift,pressed,Name,True,2022-04-14 09:59:00.674279,1649930340.674279
29,shift,pressed,Name,True,2022-04-14 09:59:00.699285,1649930340.699286


Consecutive `shift` presses are easily explainable by long presses which the OS registeres as consecutive `pressed` events. It is a modifier key.

In [41]:
big_keyboard_df.loc[big_keyboard_df["Key"] == "tab"]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
64,tab,released,Card No,True,2022-02-06 18:57:14.693870,1644173834.693871
101,tab,released,CVC,True,2022-02-06 18:57:24.125922,1644173844.125922
108,tab,released,Exp m,True,2022-02-06 18:57:25.602162,1644173845.602162
113,tab,released,Exp y,True,2022-02-06 18:57:26.707177,1644173846.707177
54,tab,released,Card No,True,2022-02-06 18:57:33.026100,1644173853.0261
...,...,...,...,...,...,...
32,tab,released,Card No,True,2022-04-05 09:42:20.966870,1649151740.966871
65,tab,released,CVC,True,2022-04-05 09:42:45.964544,1649151765.964545
72,tab,released,Exp m,True,2022-04-05 09:42:53.534079,1649151773.53408
77,tab,released,Exp y,True,2022-04-05 09:42:57.165202,1649151777.165202


For the `tab` key, I cannot find any reasonable explaination just by looking at the data.
The paper mentiones using Kivy library for event logging. It also mentiones that `tab` data is unusable and, therefore, dropped by them. It is reasonable considering they only calculated `Dwell Time` (length of key press) and `Flight Time` (time between release and press of another key). Unpaired `tab` data is useless for that features. However, it remains to be elucidated if it is not worth keeping at all as it may be of discriminating value (preference for `tab` over `left click` for jumping fields).

**NOTE**<br>
Research typical `Kivy` issues and quirks.

## Mouse data
That's all for the first look at the keyboard data. Let's look at the first user data on mouse events.

In [42]:
user1_mouse = pd.DataFrame(user1["false_data"]["test_1"]["mouse_events"]).astype({
    "Movement ID": "Int64",
    "Timestamp": "datetime64[us]",
    "Epoch": "Float64"
})

display(user1_mouse)
display(user1_mouse.describe(include='all'))
display(user1_mouse.dtypes)

,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
0,movement,"[386.0, 91.0]",2022-02-06 19:25:28.359473,1644175528.359473,1,NaN
1,left release,"[386.0, 91.00000000000001]",2022-02-06 19:25:28.408747,1644175528.408747,<NA>,NaN
2,movement,"[386.0, 92.0]",2022-02-06 19:25:28.421404,1644175528.421404,1,NaN
3,movement,"[387.0, 92.0]",2022-02-06 19:25:28.471485,1644175528.471485,1,NaN
4,movement,"[387.0, 93.0]",2022-02-06 19:25:28.496569,1644175528.496569,1,NaN
...,...,...,...,...,...,...
456,movement,"[373.0, 115.0]",2022-02-06 19:25:57.417238,1644175557.417238,11,NaN
457,movement,"[373.0, 113.0]",2022-02-06 19:25:57.443276,1644175557.443276,11,NaN
458,movement,"[374.0, 113.0]",2022-02-06 19:25:57.468304,1644175557.468304,11,NaN
459,left press,"[374.0, 113.00000000000001]",2022-02-06 19:25:57.517562,1644175557.517563,<NA>,NaN


,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
count,460,460,460,460.0,446.0,1.0
unique,3,449,NaN,<NA>,<NA>,NaN
top,movement,"[208.0, 94.0]",NaN,<NA>,<NA>,NaN
freq,446,4,NaN,<NA>,<NA>,NaN
mean,NaN,NaN,2022-02-06 19:25:41.622957,1644175541.622957,4.553812,0.0
min,NaN,NaN,2022-02-06 19:25:28.359473,1644175528.359473,1.0,0.0
25%,NaN,NaN,2022-02-06 19:25:31.510268,1644175531.510269,1.0,0.0
50%,NaN,NaN,2022-02-06 19:25:38.197915,1644175538.197916,2.0,0.0
75%,NaN,NaN,2022-02-06 19:25:52.593991,1644175552.593991,9.0,0.0
max,NaN,NaN,2022-02-06 19:25:57.517562,1644175557.517563,11.0,0.0


Event                      str
Coordinates             object
Timestamp       datetime64[us]
Epoch                  Float64
Movement ID              Int64
false_enters           float64
dtype: object

In [43]:
type(user1_mouse["Coordinates"].iloc[0])

list

Here is what I can tell from the first look at one user data.

The event data has the following structure:
- `Event` - event type, `movement` for mouse movement data and other for mouse clicks etc.
- `Coordinates` -  `[X, Y]` coordinate tuple (parsed into DataFrame as `list` type)
- `Timestamp` and `Epoch` - datetime info (see above for keyboard events)
- `Movement ID` - the counter ID of the movement, reset in each test session (according to the paper, defined as movement separated by 0.5s idle)
- `false_enters` - cannot determine from the data structure what it represents, the paper does not mention it (**NOTE** A strong candidate for dropping)

Similarly to the keyboard data, there are some issues with event logging. For starters, the second event is `left release` with no press event preceding it.

Let's now load the whole keyboard dataset, simply concatenating the data of each user.

In [54]:
dfs = []
for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for no in range(1, 11):
            dfs.append(pd.DataFrame(user_data["true_data"][f"test_{no}"]["mouse_events"]))
            dfs.append(pd.DataFrame(user_data["false_data"][f"test_{no}"]["mouse_events"]))

big_mouse_df = pd.concat(dfs)

display(big_mouse_df.isna().sum())

Event             1760
Coordinates       1760
Timestamp         1760
Epoch             1760
Movement ID      25320
false_enters    706984
dtype: int64

- We can see exactly **1760** `NA` values for first 4 columns. The number corresponds to **88 users x 20 tests**. 
- There is also an enormous amount of `false_enters` NA vals and an order of magnitude less for `Movement ID` (corresponding to events different than movements).

Let's make sure that all these **1760** `NA` values are the same rows for each of the first 4 features and see what values the remaining features exhibit.

In [55]:
nas = big_mouse_df.loc[big_mouse_df["Event"].isna()]
print(nas.isna().sum())
nas.describe(include="all")

Event           1760
Coordinates     1760
Timestamp       1760
Epoch           1760
Movement ID     1760
false_enters       0
dtype: int64


,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
count,0,0,0,0,0.0,1760.000000
unique,0,0,0,0,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,0.278409
std,NaN,NaN,NaN,NaN,NaN,1.495793
min,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,0.000000
50%,NaN,NaN,NaN,NaN,NaN,0.000000
75%,NaN,NaN,NaN,NaN,NaN,0.000000


It looks like only `false_enters` has some value for these rows. Considering that it is unknown what this feature represents, I will drop it in the loading function before more detailed analysis and feature engineering.

In [46]:
big_mouse_df.loc[big_mouse_df["Movement ID"].isna()]

,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
55,left press,"[307.0, 383.00000000000006]",2022-02-06 18:57:08.782240,1644173828.7822409,NaN,NaN
56,left release,"[307.0, 383.00000000000006]",2022-02-06 18:57:08.856246,1644173828.8562467,NaN,NaN
74,left press,"[318.0, 312.0]",2022-02-06 18:57:09.122411,1644173829.1224115,NaN,NaN
78,left release,"[319.0, 305.0]",2022-02-06 18:57:09.186461,1644173829.186461,NaN,NaN
83,left press,"[324.0, 291.0]",2022-02-06 18:57:09.428796,1644173829.428797,NaN,NaN
...,...,...,...,...,...,...
160,left release,"[422.0, 380.99999999999994]",2022-04-05 09:41:55.375268,1649151715.375268,NaN,NaN
191,left press,"[288.0, 285.0]",2022-04-05 09:41:58.584616,1649151718.5846167,NaN,NaN
192,left release,"[288.0, 285.0]",2022-04-05 09:41:58.812721,1649151718.8127215,NaN,NaN
294,left press,"[376.0, 88.99999999999999]",2022-04-05 09:43:08.412875,1649151788.4128754,NaN,NaN


Here I confirm that missing `Movement ID` values correspond to events different from `movement`.

In [47]:
big_mouse_df.drop(["false_enters"], axis=1, inplace=True)
big_mouse_df.dropna(subset=["Event"], inplace=True)

In [48]:
big_mouse_df.isna().sum()

Event              0
Coordinates        0
Timestamp          0
Epoch              0
Movement ID    23560
dtype: int64

In [49]:
big_mouse_df["Event"].unique()

<ArrowStringArray>
[           'movement',          'left press',        'left release',
    'scrolldown press',  'scrolldown release',      'scrollup press',
    'scrollup release',   'scrollright press', 'scrollright release',
         'right press',       'right release',    'scrollleft press',
  'scrollleft release',        'mouse4 press',      'mouse4 release']
Length: 15, dtype: str

There are multiple moouse events registered:
- button clicks: `left press/release`, `right press/release`, `mouse4 press/release`
- scrolling: `scrollup/scrolldown/scrollleft/scrollright press/release`
- mouse movement: `movement`

I am particularly curious about the distribution of movement count per test.

In [50]:
movement_counts = []

for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for test_no in range(1, 11):
            true_df = pd.DataFrame(user_data["true_data"][f"test_{test_no}"]["mouse_events"])
            movement_counts.append(true_df["Movement ID"].nunique())

            false_df = pd.DataFrame(user_data["false_data"][f"test_{test_no}"]["mouse_events"])
            movement_counts.append(false_df["Movement ID"].nunique())

movement_counts_series = pd.Series(movement_counts).rename("Movement Count")
movement_counts_series.agg([np.mean, np.std, np.min, np.max, np.median])

mean       7.007955
std        5.108810
min        1.000000
max       73.000000
median     7.000000
Name: Movement Count, dtype: float64

Let's start constructing the data loading function based on what I already know about the data.

I will drop `Timestamp` and `false_enters` cols from mouse data and `Timestamp` from keyboard data. It is also necessary to ensure the dtype of `Epoch` as `float`.

***Why did I choose `Epoch` field over `Timestamp`?***<br>
Since the feature engineering process primarily involves calculating time intervals between events rather than manipulating calendar dates, the Epoch representation was selected.
Using `float64` values allows time differences to be calculated directly in seconds, simplifying the implementation and avoiding repeated `datetime`-to-`timedelta` conversions.
Although floating-point arithmetic introduces minor precision limitations, these are negligible relative to the time intervals analyzed in this project.

In [59]:
mouse_data = {}

for user_file in json_file_list:
    with open(raw_kmt_dataset / user_file, 'r') as f:
        user_data = json.load(f)
        for test_no in range(1, 11):
            mouse_true = user_data["true_data"][f"test_{test_no}"]["mouse_events"]
            for event_id, event in enumerate(mouse_true):
                mouse_data[(int(user_file.split("_")[-1].split(".")[0]), True, test_no, event_id)] = event 

            mouse_false = user_data["false_data"][f"test_{test_no}"]["mouse_events"]
            for event_id, event in enumerate(mouse_false):
                mouse_data[(int(user_file.split("_")[-1].split(".")[0]), False, test_no, event_id)] = event


mouse_df = pd.DataFrame.from_dict(mouse_data, orient='index').astype({
    "Movement ID": "Int16",
    "Epoch": "float64"
})
mouse_df.index.names = [
    "user_id",
    "is_genuine",
    "test_id",
    "event_id",
]
mouse_df.drop(["false_enters", "Timestamp"], axis=1, inplace=True)

mouse_df.head()

Event     Coordinates         Epoch  \
user_id is_genuine test_id event_id                                           
1       True       1       0         movement   [30.0, 581.0]  1.644174e+09   
                           1         movement   [76.0, 569.0]  1.644174e+09   
                           2         movement  [159.0, 549.0]  1.644174e+09   
                           3         movement  [194.0, 540.0]  1.644174e+09   
                           4         movement  [239.0, 531.0]  1.644174e+09   

                                     Movement ID  
user_id is_genuine test_id event_id               
1       True       1       0                   1  
                           1                   1  
                           2                   1  
                           3                   1  
                           4                   1